# Customer Behaviour Analysis
Predicts whether a user session will end in a purchase using session-level features.


## 1. Setup & Data Download

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import os

os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!pip install kaggle shap xgboost --quiet

In [ ]:
!kaggle datasets download -d mkechinov/ecommerce-behavior-data-from-multi-category-store
!unzip -q ecommerce-behavior-data-from-multi-category-store.zip

## 2. Load & Optimise Memory

In [ ]:
import pandas as pd

df = pd.read_csv('2019-Oct.csv', nrows=500000)

# Downcast to reduce memory
df['event_type']    = df['event_type'].astype('category')
df['category_code'] = df['category_code'].astype('category')
df['brand']         = df['brand'].astype('category')
df['product_id']    = pd.to_numeric(df['product_id'],  downcast='integer')
df['category_id']   = pd.to_numeric(df['category_id'], downcast='integer')
df['user_id']       = pd.to_numeric(df['user_id'],     downcast='integer')
df['price']         = pd.to_numeric(df['price'],       downcast='float')
df['event_time']    = pd.to_datetime(df['event_time'])

print(df.shape)
df.head()

## 3. Exploratory Data Analysis

In [ ]:
print('Nulls:\n', df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())
print('\nUnique users:', df['user_id'].nunique())
print('Unique sessions:', df['user_session'].nunique())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df['event_type'].value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Event Type Distribution')
axes[0].set_xlabel('Event Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

df['hour'] = df['event_time'].dt.hour
df['hour'].value_counts().sort_index().plot(kind='line', marker='o', ax=axes[1])
axes[1].set_title('Shopping Activity by Hour')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Number of Events')

plt.tight_layout()
plt.show()

In [ ]:
# Conversion funnel
event_counts = df['event_type'].value_counts()
views, carts, purchases = event_counts['view'], event_counts['cart'], event_counts['purchase']

print(f'View → Cart:     {carts / views * 100:.2f}%')
print(f'Cart → Purchase: {purchases / carts * 100:.2f}%')
print(f'View → Purchase: {purchases / views * 100:.2f}%')

In [ ]:
# Top categories by events vs purchases
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

df['category_code'].value_counts().head(10).plot(kind='bar', ax=axes[0])
axes[0].set_title('Top 10 Categories by Events')
axes[0].tick_params(axis='x', rotation=75)

df[df['event_type'] == 'purchase']['category_code'].value_counts().head(10).plot(kind='bar', ax=axes[1])
axes[1].set_title('Top 10 Categories by Purchases')
axes[1].tick_params(axis='x', rotation=75)

plt.tight_layout()
plt.show()

## 4. Feature Engineering
All features are aggregated at the **session** level. No purchase-derived signals are included as features (that would be leakage).

In [ ]:
# --- Bot / outlier session filtering ---
# Sessions with an extreme number of events in a short window are likely bots.
session_event_counts = df.groupby('user_session').size()
upper_limit = session_event_counts.quantile(0.999)
valid_sessions = session_event_counts[session_event_counts <= upper_limit].index
df = df[df['user_session'].isin(valid_sessions)]
print(f'Sessions after bot filter: {df["user_session"].nunique()} (removed {len(session_event_counts) - len(valid_sessions)} outlier sessions)')

In [ ]:
# Sort once for time-based computations
df = df.sort_values(by=['user_session', 'event_time'])

In [ ]:
# --- Session time bounds ---
session_time = df.groupby('user_session')['event_time'].agg(['min', 'max'])
session_time['session_duration'] = (
    session_time['max'] - session_time['min']
).dt.total_seconds()

In [ ]:
# --- Target label ---
purchase_sessions = df[df['event_type'] == 'purchase']['user_session'].unique()
session_time['purchased'] = session_time.index.isin(purchase_sessions)
session_time['target']    = session_time['purchased'].astype(int)

print('Class distribution:')
print(session_time['target'].value_counts())
print(f'Purchase rate: {session_time["target"].mean() * 100:.2f}%')

In [ ]:
# --- Event-type counts per session ---
for event in ['view', 'cart']:
    counts = df[df['event_type'] == event].groupby('user_session').size()
    session_time[f'{event}_count'] = counts
    session_time[f'{event}_count'] = session_time[f'{event}_count'].fillna(0)

session_time['session_length']   = df.groupby('user_session').size()
session_time['unique_products']  = df.groupby('user_session')['product_id'].nunique()

In [ ]:
# --- Ratio features ---
session_time['cart_view_ratio'] = (
    session_time['cart_count'] / session_time['view_count']
).replace([float('inf')], 0).fillna(0)

In [ ]:
# --- Average time gap between events ---
df['time_diff'] = (
    df.groupby('user_session')['event_time']
    .diff()
    .dt.total_seconds()
)
session_time['avg_time_gap'] = (
    df.groupby('user_session')['time_diff'].mean()
).fillna(0)

In [ ]:
# --- Repeat product views ---
total_views   = df[df['event_type'] == 'view'].groupby(['user_session', 'product_id']).size()
repeated_views = total_views[total_views > 1]
repeat_count  = repeated_views.groupby('user_session').size()
session_time['repeat_product_views'] = repeat_count.fillna(0)

In [ ]:
# --- Category diversity ---
# FIX: fillna(0) added — sessions with all-null category_code would produce NaN
# without this, Logistic Regression and Random Forest will fail silently.
unique_categories = df.groupby('user_session')['category_code'].nunique()
session_time['unique_categories'] = unique_categories.fillna(0)

In [ ]:
# --- Price features (new) ---
# avg_price: purchase intent correlates with the price tier the user is browsing.
# max_price: captures interest in high-value items.
# price_std: variance indicates comparison shopping behaviour.
price_stats = df.groupby('user_session')['price'].agg(
    avg_price='mean',
    max_price='max',
    price_std='std'
).fillna(0)
session_time = session_time.join(price_stats)

In [ ]:
# --- Day-of-week feature (new) ---
# Mode of event day within session. Weekend vs weekday is a known purchase-intent signal.
df['dayofweek'] = df['event_time'].dt.dayofweek
session_time['dayofweek'] = (
    df.groupby('user_session')['dayofweek']
    .agg(lambda x: x.mode().iloc[0])
)

In [ ]:
print(session_time.isnull().sum())
session_time.head()

## 5. Modelling with Hyperparameter Tuning (RandomizedSearchCV)

In [ ]:
features = [
    'session_duration',
    'session_length',
    'unique_products',
    'cart_count',
    'view_count',
    'cart_view_ratio',
    'avg_time_gap',
    'repeat_product_views',
    'unique_categories',
    'avg_price',
    'max_price',
    'price_std',
    'dayofweek',
]

X = session_time[features]
y = session_time['target']

print(f'Feature matrix: {X.shape}, class balance: {y.mean() * 100:.2f}% positive')

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

lr_param_dist = {
    'C':       loguniform(1e-3, 1e2),   # inverse regularisation strength
    'solver':  ['lbfgs', 'saga'],
    'penalty': ['l2'],
}

lr_search = RandomizedSearchCV(
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    param_distributions=lr_param_dist,
    n_iter=20,
    scoring='roc_auc',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
lr_search.fit(X_train_scaled, y_train)

print('Best LR params:', lr_search.best_params_)
print(f'Best CV ROC-AUC: {lr_search.best_score_:.4f}')

lr_best = lr_search.best_estimator_
lr_pred = lr_best.predict(X_test_scaled)
lr_prob = lr_best.predict_proba(X_test_scaled)[:, 1]

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_param_dist = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [6, 8, 10, 12, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', 0.5],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42),
    param_distributions=rf_param_dist,
    n_iter=20,
    scoring='roc_auc',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
rf_search.fit(X_train, y_train)

print('Best RF params:', rf_search.best_params_)
print(f'Best CV ROC-AUC: {rf_search.best_score_:.4f}')

rf_best = rf_search.best_estimator_
rf_pred = rf_best.predict(X_test)
rf_prob = rf_best.predict_proba(X_test)[:, 1]

In [ ]:
from xgboost import XGBClassifier

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

xgb_param_dist = {
    'n_estimators':   [100, 200, 300],
    'max_depth':      [3, 4, 5, 6, 7],
    'learning_rate':  [0.01, 0.05, 0.1, 0.2],
    'subsample':      [0.6, 0.7, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma':          [0, 0.1, 0.3, 0.5],
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(
        scale_pos_weight=neg / pos,
        random_state=42,
        eval_metric='logloss',
        verbosity=0
    ),
    param_distributions=xgb_param_dist,
    n_iter=30,
    scoring='roc_auc',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
xgb_search.fit(X_train, y_train)

print('Best XGB params:', xgb_search.best_params_)
print(f'Best CV ROC-AUC: {xgb_search.best_score_:.4f}')

xgb_best = xgb_search.best_estimator_
xgb_pred = xgb_best.predict(X_test)
xgb_prob = xgb_best.predict_proba(X_test)[:, 1]

## 6. Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report
)

# Print best params summary before metrics
print('=== Best Hyperparameters ===')
print(f'LR:  {lr_search.best_params_}')
print(f'RF:  {rf_search.best_params_}')
print(f'XGB: {xgb_search.best_params_}')
print()

# --- Side-by-side model comparison (tuned models) ---
model_results = {
    'Logistic Regression (tuned)': (lr_pred, lr_prob),
    'Random Forest (tuned)':       (rf_pred, rf_prob),
    'XGBoost (tuned)':             (xgb_pred, xgb_prob),
}

rows = []
for name, (pred, prob) in model_results.items():
    rows.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, pred), 4),
        'Precision': round(precision_score(y_test, pred), 4),
        'Recall':    round(recall_score(y_test, pred), 4),
        'F1':        round(f1_score(y_test, pred), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, prob), 4),
    })

comparison_df = pd.DataFrame(rows).set_index('Model')
print('=== Test Set Performance ===')
print(comparison_df.to_string())

In [ ]:
# --- Confusion matrices ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

import numpy as np

for ax, (name, (pred, _)) in zip(axes, model_results.items()):
    cm = confusion_matrix(y_test, pred)
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['No Purchase', 'Purchase'])
    ax.set_yticklabels(['No Purchase', 'Purchase'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')

plt.tight_layout()
plt.show()

In [ ]:
# --- ROC Curves ---
from sklearn.metrics import roc_curve

plt.figure(figsize=(7, 5))
colors = ['steelblue', 'darkorange', 'green']

for (name, (_, prob)), color in zip(model_results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color)

plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Precision-Recall Curves ---
# PR curve is more informative than ROC when classes are heavily imbalanced.
from sklearn.metrics import precision_recall_curve, average_precision_score

plt.figure(figsize=(7, 5))

for (name, (_, prob)), color in zip(model_results.items(), colors):
    precision, recall, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)
    plt.plot(recall, precision, label=f'{name} (AP={ap:.3f})', color=color)

baseline = y_test.mean()
plt.axhline(y=baseline, color='k', linestyle='--', label=f'Baseline ({baseline:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Cross-validation on best XGBoost model ---
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_best, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f'XGBoost (tuned) 5-Fold CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Per-fold: {[round(s, 4) for s in cv_scores]}')

## 7. Feature Importance & SHAP

In [ ]:
# --- XGBoost (tuned) feature importance ---
xgb_fi = pd.DataFrame({
    'Feature':    features,
    'Importance': xgb_best.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(9, 5))
plt.barh(xgb_fi['Feature'], xgb_fi['Importance'])
plt.xlabel('Importance Score')
plt.title('XGBoost (tuned) Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# --- SHAP summary plot ---
# SHAP shows the direction and magnitude of each feature's effect,
# not just its global rank.
import shap

explainer   = shap.TreeExplainer(xgb_best)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test, feature_names=features, plot_type='dot')

## 8. Export Model for FastAPI Deployment
Saves the XGBoost model + scaler as a single `.pkl` bundle and downloads it.
The FastAPI server script is also written to disk and downloaded.

### Expected folder structure after download
```
app/
├── main.py          ← FastAPI server
├── model_bundle.pkl ← model + scaler + feature list
└── requirements.txt
```

In [ ]:
import joblib

# Bundle everything the server needs into one file
model_bundle = {
    'model':       xgb_best,
    'scaler':      scaler,
    'features':    features,
    'model_type':  'xgboost',
    'best_params': xgb_search.best_params_,
    'cv_auc':      round(xgb_search.best_score_, 4),
}

joblib.dump(model_bundle, 'model_bundle.pkl')
print('Saved model_bundle.pkl')
print(f'Bundled params: {xgb_search.best_params_}')

In [ ]:
# Write the FastAPI server to disk so it can be downloaded alongside the model
fastapi_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import joblib
import pandas as pd
import numpy as np

app = FastAPI(title="Customer Purchase Predictor", version="1.0.0")

bundle   = joblib.load("model_bundle.pkl")
model    = bundle["model"]
features = bundle["features"]


class SessionFeatures(BaseModel):
    session_duration:      float
    session_length:        float
    unique_products:       float
    cart_count:            float
    view_count:            float
    cart_view_ratio:       float
    avg_time_gap:          float
    repeat_product_views:  float
    unique_categories:     float
    avg_price:             float
    max_price:             float
    price_std:             float
    dayofweek:             float


class PredictionResponse(BaseModel):
    will_purchase:      bool
    purchase_probability: float


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict", response_model=PredictionResponse)
def predict(session: SessionFeatures):
    try:
        input_df = pd.DataFrame([session.dict()])[features]
        prob     = float(model.predict_proba(input_df)[0][1])
        return PredictionResponse(
            will_purchase=prob >= 0.5,
            purchase_probability=round(prob, 4)
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/predict/batch")
def predict_batch(sessions: list[SessionFeatures]):
    try:
        input_df = pd.DataFrame([s.dict() for s in sessions])[features]
        probs    = model.predict_proba(input_df)[:, 1]
        return [
            {"will_purchase": bool(p >= 0.5), "purchase_probability": round(float(p), 4)}
            for p in probs
        ]
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
'''.strip()

requirements = '''fastapi
uvicorn[standard]
xgboost
scikit-learn
pandas
joblib
numpy
'''.strip()

with open('main.py', 'w') as f:
    f.write(fastapi_code)

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print('Written main.py and requirements.txt')

In [ ]:
# Download all three files
from google.colab import files

files.download('model_bundle.pkl')
files.download('main.py')
files.download('requirements.txt')

print('Downloads triggered.')

## 9. Running the Server Locally

After downloading, place all three files in the same folder and run:

```bash
pip install -r requirements.txt
uvicorn main:app --reload
```

The interactive docs will be at **http://127.0.0.1:8000/docs**

### Example request

```bash
curl -X POST http://127.0.0.1:8000/predict \
  -H "Content-Type: application/json" \
  -d '{
    "session_duration": 320,
    "session_length": 12,
    "unique_products": 5,
    "cart_count": 3,
    "view_count": 9,
    "cart_view_ratio": 0.33,
    "avg_time_gap": 28.5,
    "repeat_product_views": 2,
    "unique_categories": 2,
    "avg_price": 149.99,
    "max_price": 399.0,
    "price_std": 82.4,
    "dayofweek": 5
  }'
```

### Expected response

```json
{"will_purchase": true, "purchase_probability": 0.7821}
```